<a href="https://colab.research.google.com/github/zeroKool1ne/ironhack_langchain/blob/main/01_langchain_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from IPython.display import HTML, display

def set_css():
    display(HTML('''
    <style>
      pre {
          white-space: pre-wrap;
      }
    </style>
    '''))

get_ipython().events.register('pre_run_cell', set_css)

# LangChain Memory

Most common use-case for building an AI application with LangChain is a chatbot trained on a knowledgebase. But there is a problem with just using a RetrievalQA method i.e the chatbot developing using this method has no memory and thus cannot answer in a conversational manner

For example if you check the below flow

**Human** : How to add a video funnel

**AI**    : If you want to set up a video funnel, these are the steps you need to follow:

Step 1- First of all, go to the Marketing tab at the video level.

Step 2- Click on the “Add new funnel” button.

Step 3- Set the title to grab the users’ attention.

Step 4- Set the text for the action that you want would be taken by the users.

Step 5- Select the video for the Funnel.

Step 6- Set the start time at which the pop-up would appear to the user.

Step 7- Once you are satisfied with all the things, click on the “Save Changes” button.

**Human** : Now tell only 5 steps from above

**AI**.   : Sorry I am unable to answer, can you provide more context

As you can see, the chatbot has no memory and thus unable to answer any follow up questions. To solve this we will be using Memory


The memory allows a Large Language Model (LLM) to remember previous interactions with the user. By default, LLMs are stateless — meaning each incoming query is processed independently of other interactions. The only thing that exists for a stateless agent is the current input, nothing else.

There are many applications where remembering previous interactions is very important, such as chatbots. Conversational memory allows us to do that.

There are several ways that we can implement conversational memory. In the context of LangChain, they are all built on top of the `ConversationChain`.

### Types of Memory

Let's discuss some of the most popular memory methods available in Langchain.

1. ConversationBufferMemory
2. ConversationSummaryMemory
3. ConversationBufferWindowMemory
4. ConversationSummaryBufferMemory

We shall discuss each of these with an example

### ConversationBufferMemory

This is a simple method that involves storing every chat interaction directly in the buffer. Although it provides good results, it has few drawbacks

1. Because every message is stored, the amount of data being sent to api is high and thus results in higher costs and slower speed of response
2. ChatGPT has input context limit which can get crossed with few messages and thus can result in error

Let's understand it's working with the help of an example. We will try adding memory to a chain

In [2]:
!pip install -q langchain langchain-classic langchain-anthropic anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.3 MB/s eta 0:00:00


In [3]:
!pip install -q langchain langchain-anthropic anthropic

import os
from google.colab import userdata

CLAUDE_API_KEY = userdata.get('claude_ironhack')
os.environ["ANTHROPIC_API_KEY"] = CLAUDE_API_KEY

from langchain_anthropic import ChatAnthropic
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationSummaryMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryBufferMemory,
)

llm = ChatAnthropic(model="claude-haiku-4-5", api_key=CLAUDE_API_KEY, temperature=0.7)

In [4]:
# Initialising conversation chain
conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

/tmp/ipykernel_735/3942203023.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory=ConversationBufferMemory()
/tmp/ipykernel_735/3942203023.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(


We can see the prompt template used by the `ConversationChain` like so:

In [5]:
print(conversation.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


Here, the prompt primes the model by telling it that the following is a conversation between a human (us) and an AI (`gpt3-turbo`). The prompt attempts to reduce hallucinations (where a model makes things up) by stating:

**"If the AI does not know the answer to a question, it truthfully says it does not know."**

This can help but does not solve the problem of hallucinations — but we will save this for the topic of a future chapter.

Following the initial prompt, we see two parameters; `{history}` and `{input}`. The `{input}` is where we’d place the latest human query;

The `{history}` is where conversational memory is used. Here, we feed in information about the conversation history between the human and AI.

These two parameters — `{history}` and `{input}` — are passed to the LLM within the prompt template we just saw, and the output that we (hopefully) return is simply the predicted continuation of the conversation.



In [ ]:
# only relevant if you would wanna skip verbose and initialise conversation  again withou it
#conversation_buf = ConversationChain(
#   llm=llm,
#    memory=ConversationBufferMemory()
#)

In [7]:
conversation.invoke("Good morning AI!")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Good morning AI!
AI:

> Finished chain.


{'input': 'Good morning AI!',
 'history': '',
 'response': "Good morning! It's wonderful to hear from you! I hope you're having a great day so far. I'm here and ready to chat about whatever's on your mind—whether that's answering questions, brainstorming ideas, having a thoughtful discussion, or just having a friendly conversation. \n\nIs there anything specific you'd like to talk about today, or would you just like to see where the conversation goes? I'm genuinely curious what brings you here! 😊"}

We return the first response from the conversational agent. Let’s continue the conversation, writing prompts that the LLM can only answer if it considers the conversation history. We also add a count_tokens function so we can see how many tokens are being used by each interaction.

In [8]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

def count_tokens(chain, query):
    callback = UsageMetadataCallbackHandler()
    result = chain.run(query, callbacks=[callback])

    total_tokens = sum(v["total_tokens"] for v in callback.usage_metadata.values())
    print(f'Spent a total of {total_tokens} tokens')

    return result

In [9]:
count_tokens(
    conversation,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

/tmp/ipykernel_735/2204244037.py:5: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = chain.run(query, callbacks=[callback])




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Good morning AI!
AI: Good morning! It's wonderful to hear from you! I hope you're having a great day so far. I'm here and ready to chat about whatever's on your mind—whether that's answering questions, brainstorming ideas, having a thoughtful discussion, or just having a friendly conversation. 

Is there anything specific you'd like to talk about today, or would you just like to see where the conversation goes? I'm genuinely curious what brings you here! 😊
Human: My interest here is to explore the potential of integrating Large Language Models with external knowledge
AI:

> Finished chain.
Spent a total of 525 tokens


"What a fascinating area to explore! Integrating Large Language Models with external knowledge is genuinely one of the most exciting frontiers in AI right now, and there are so many compelling directions this can take.\n\nLet me share some of the key approaches and opportunities I'm aware of:\n\n**Retrieval-Augmented Generation (RAG)** is probably the most practical method being used today. This involves having an LLM query an external knowledge base or database to retrieve relevant information before generating a response. So instead of relying solely on what's in the model's training data, it can pull in current, specific, or specialized information. This is especially useful for things like medical databases, legal documents, or real-time information.\n\n**Knowledge Graphs** are another powerful integration point. By connecting an LLM to a structured knowledge graph (like those used in semantic web technologies), you can give the model access to explicit relationships between concep

In [10]:
count_tokens(
    conversation,
    "I just want to analyze the different possibilities. What can you think of?"
)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Good morning AI!
AI: Good morning! It's wonderful to hear from you! I hope you're having a great day so far. I'm here and ready to chat about whatever's on your mind—whether that's answering questions, brainstorming ideas, having a thoughtful discussion, or just having a friendly conversation. 

Is there anything specific you'd like to talk about today, or would you just like to see where the conversation goes? I'm genuinely curious what brings you here! 😊
Human: My interest here is to explore the potential of integrating Large Language Models with external knowledge
AI: What a fascinating area to explore! Integrating Large Language Models with external knowledg

"Excellent! Let me break down the different possibilities across several dimensions:\n\n**By Integration Architecture:**\n\n1. **Retrieval-Augmented Generation (RAG)** - Query external sources in real-time before generating responses. Great for currency and accuracy.\n\n2. **In-Context Learning** - Passing external knowledge directly in the prompt/context window. Simple but limited by token limits.\n\n3. **Fine-tuning/Continued Training** - Permanently adapting the model weights using external knowledge. More resource-intensive but deeply integrated.\n\n4. **Prompt Engineering with Knowledge** - Structuring prompts to guide the LLM using external facts. Lightweight and flexible.\n\n5. **Hybrid Approaches** - Combining multiple methods, like fine-tuning plus RAG for specialized domains.\n\n**By Knowledge Source Type:**\n\n- **Structured Data** (databases, knowledge graphs, ontologies)\n- **Unstructured Data** (documents, articles, web pages)\n- **Real-time Data** (APIs, live feeds, sens

In [ ]:
count_tokens(
    conversation,
    "What is my aim again?"
)

Spent a total of 376 tokens


'Your aim is to explore the potential of integrating Large Language Models with external knowledge to enhance their capabilities and improve their performance on specific tasks.'

The LLM can clearly remember the history of the conversation. Let’s take a look at how this conversation history is stored by the ConversationBufferMemory:

In [11]:
print(conversation.memory.buffer)

Human: Good morning AI!
AI: Good morning! It's wonderful to hear from you! I hope you're having a great day so far. I'm here and ready to chat about whatever's on your mind—whether that's answering questions, brainstorming ideas, having a thoughtful discussion, or just having a friendly conversation. 

Is there anything specific you'd like to talk about today, or would you just like to see where the conversation goes? I'm genuinely curious what brings you here! 😊
Human: My interest here is to explore the potential of integrating Large Language Models with external knowledge
AI: What a fascinating area to explore! Integrating Large Language Models with external knowledge is genuinely one of the most exciting frontiers in AI right now, and there are so many compelling directions this can take.

Let me share some of the key approaches and opportunities I'm aware of:

**Retrieval-Augmented Generation (RAG)** is probably the most practical method being used today. This involves having an LL

We can see that the buffer saves every interaction in the chat history directly. There are a few pros and cons to this approach. In short, they are:

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid #000;
            padding: 10px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>
    <table>
        <thead>
            <tr>
                <th>Pros</th>
                <th>Cons</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Storing everything gives the LLM the maximum amount of information</td>
                <td>More tokens mean slowing response times and higher costs</td>
            </tr>
            <tr>
                <td>Storing everything is simple and intuitive</td>
                <td>Long conversations cannot be remembered as we hit the LLM token limit (4096 tokens for text-davinci-003 and gpt-3.5-turbo)</td>
            </tr>
        </tbody>
    </table>
</body>
</html>


The `ConversationBufferMemory` is an excellent option to get started with but is limited by the storage of every interaction. Let’s take a look at other options that help remedy this.

### ConversationSummaryMemory
Using ConversationBufferMemory, we very quickly use a lot of tokens and even exceed the context window limit of even the most advanced LLMs available today.

To avoid excessive token usage, we can use ConversationSummaryMemory. As the name would suggest, this form of memory summarizes the conversation history before it is passed to the {history} parameter.

We initialize the ConversationChain with the summary memory like so:

In [12]:
conversation_sum = ConversationChain(
	llm=llm,
	memory=ConversationSummaryMemory(llm=llm)
)

/tmp/ipykernel_735/2372352816.py:3: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory=ConversationSummaryMemory(llm=llm)


When using `ConversationSummaryMemory`, we need to pass an LLM to the object because the summarization is powered by an LLM. We can see the prompt used to do this here:

In [13]:
print(conversation_sum.memory.prompt.template)

Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:


Using this, we can summarize every new interaction and append it to a “running summary” of all past interactions. Let’s have another conversation utilizing this approach.

In [14]:
# without count_tokens we'd call `conversation_sum("Good morning AI!")`
# but let's keep track of our tokens:
count_tokens(
    conversation_sum,
    "Good morning AI!"
)

Spent a total of 192 tokens


"Good morning! It's great to hear from you! I hope you're having a wonderful day so far. \n\nI'm here and ready to chat about pretty much anything you'd like to discuss—whether that's answering questions, exploring ideas, having a creative conversation, helping with work or projects, or just having a friendly chat. I really enjoy diving into specific details and topics, so feel free to ask me about something you're curious about or interested in.\n\nWhat's on your mind today? Is there anything particular you'd like to talk about or get help with?"

In [15]:
count_tokens(
    conversation_sum,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

Spent a total of 511 tokens


"Good morning! What a fascinating topic to explore! I'm genuinely excited about this direction.\n\nIntegrating LLMs with external knowledge is one of the most promising frontiers in AI right now, and there are so many interesting angles we could discuss. Let me share some of the key approaches and challenges I'm aware of:\n\n**Current Integration Methods:**\n- **Retrieval-Augmented Generation (RAG)**: This is probably the most practical approach being used today. Systems retrieve relevant documents or data from external sources, then feed that context into the LLM to generate more accurate, grounded responses. It helps reduce hallucinations significantly.\n- **Knowledge Graphs**: Connecting LLMs to structured knowledge graphs (like those used in semantic search) allows for more precise, relational information retrieval.\n- **Fine-tuning on Domain Data**: Training or adapting LLMs on specific external datasets to specialize them for particular fields.\n- **Tool Use & APIs**: LLMs callin

In [ ]:
count_tokens(
    conversation_sum,
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 809 tokens


'There are several ways to integrate external knowledge with Large Language Models like GPT-3. One approach is to use knowledge graphs, which organize information in a structured way that can be easily accessed by the model. Another approach is to use pre-trained embeddings, which capture semantic relationships between words and concepts. Additionally, fine-tuning the model on specific knowledge domains can also improve its performance in that area. These are just a few possibilities, but there is a lot of potential for creativity and innovation in this field. Is there a specific approach you are interested in exploring further?'

In [ ]:
count_tokens(
    conversation_sum,
    "Which data source types could be used to give context to the model?"
)

Spent a total of 851 tokens


"There are several types of data sources that can be used to give context to the model. Some common ones include structured data from databases, unstructured data from text documents or websites, knowledge graphs that represent relationships between entities, and domain-specific data from specialized sources. Each type of data source can provide unique insights and information to help enhance the model's understanding and performance. Is there a specific type of data source you are interested in learning more about?"

In [ ]:
count_tokens(
    conversation_sum,
    "What is my aim again?"
)

Spent a total of 862 tokens


'Your aim is to explore the potential of integrating Large Language Models with external knowledge to enhance their capabilities in natural language processing.'

In this case the summary contains enough information for the LLM to “remember” our original aim. We can see this summary in it’s raw form like so:

In [18]:
print(conversation_sum.memory.buffer)

The human greets the AI good morning. The AI responds warmly, expressing enthusiasm about the conversation and offering to help with a wide range of topics. The human expresses interest in exploring the potential of integrating Large Language Models with external knowledge. The AI shares enthusiasm for this topic and provides a comprehensive overview of current integration methods including Retrieval-Augmented Generation (RAG), Knowledge Graphs, fine-tuning on domain data, and tool use with APIs. The AI explains key benefits such as access to current information, domain-specific accuracy, reduced hallucinations, and source transparency. The AI also acknowledges challenges including integration latency, computational costs, managing inconsistencies between LLM training and external sources, and context window limitations. The AI then asks the human which aspects interest them most—whether they're thinking about technical implementation, specific use cases, or broader implications.


The number of tokens being used for this conversation is greater than when using the `ConversationBufferMemory`, so is there any advantage to using `ConversationSummaryMemory` over the buffer memory?

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/token_interaction.webp" alt='auto' width="1000"/>
</div>

Token count (y-axis) for the buffer memory vs. summary memory as the number of interactions (x-axis) increases.

For longer conversations. As shown above, the summary memory initially uses far more tokens. However, as the conversation progresses, the summarization approach grows more slowly. In contrast, the buffer memory continues to grow linearly with the number of tokens in the chat.

We can summarize the pros and cons of ConversationSummaryMemory as follows:

We can summarize the pros and cons of `ConversationSummaryMemory` as follows:

<!DOCTYPE html>
<html>
<head>
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid black;
            padding: 8px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>

<table>
    <tr>
        <th>Pros</th>
        <th>Cons</th>
    </tr>
    <tr>
        <td>Shortens the number of tokens for long conversations.</td>
        <td>Can result in higher token usage for smaller conversations.</td>
    </tr>
    <tr>
        <td>Enables much longer conversations.</td>
        <td>Memorization of the conversation history is wholly reliant on the summarization ability of the intermediate summarization LLM.</td>
    </tr>
    <tr>
        <td>Relatively straightforward implementation, intuitively simple to understand.</td>
        <td>Also requires token usage for the summarization LLM; this increases costs (but does not limit conversation length).</td>
    </tr>
</table>

</body>
</html>


Conversation summarization is a good approach for cases where long conversations are expected. Yet, it is still fundamentally limited by token limits. After a certain amount of time, we still exceed context window limits - Maybe NOT, depending on the LLM being used.

### ConversationBufferWindowMemory
The `ConversationBufferWindowMemor`y acts in the same way as our earlier “buffer memory” but adds a window to the memory. Meaning that we only keep a given number of past interactions before “forgetting” them. We use it like so:

In [20]:
conversation_bufw = ConversationChain(
	llm=llm,
	verbose=True,
	memory=ConversationBufferWindowMemory(k=1)
)

In this instance, we set `k=1` — this means the window will remember the single latest interaction between the human and AI. That is the latest human response and the latest AI response. We can see the effect of this below:

In [21]:
count_tokens(
    conversation_bufw,
    "Good morning AI!"
)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Good morning AI!
AI:

> Finished chain.
Spent a total of 190 tokens


"Good morning! It's great to hear from you! I hope you're having a wonderful day so far. \n\nI'm here and ready to chat about pretty much anything you'd like—whether that's answering questions, having a creative conversation, working through a problem, or just having a friendly chat. I can dive into specific details about topics I know about, or I'm happy to admit when something is outside my knowledge.\n\nWhat's on your mind today? Is there anything I can help you with, or would you just like to have a nice conversation?"

In [22]:
count_tokens(
    conversation_bufw,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Good morning AI!
AI: Good morning! It's great to hear from you! I hope you're having a wonderful day so far. 

I'm here and ready to chat about pretty much anything you'd like—whether that's answering questions, having a creative conversation, working through a problem, or just having a friendly chat. I can dive into specific details about topics I know about, or I'm happy to admit when something is outside my knowledge.

What's on your mind today? Is there anything I can help you with, or would you just like to have a nice conversation?
Human: My interest here is to explore the potential of integrating Large Language Models with external knowledge
AI:

> Finish

"Oh, that's a fascinating area! I'm genuinely excited to discuss this with you—it's one of the most active and important frontiers in AI right now.\n\nThere are several really compelling approaches to integrating LLMs with external knowledge:\n\n**Retrieval-Augmented Generation (RAG)** is probably the most popular approach. The basic idea is that when you query an LLM, it first retrieves relevant documents or data from an external knowledge base, then uses those retrieved chunks to inform its response. This helps ground the model's outputs in factual, up-to-date information rather than relying solely on what's in its training data. Tools like LangChain and LlamaIndex have made implementing RAG much more accessible.\n\n**Knowledge Graphs** are another powerful integration point. Rather than just retrieving raw text, you can structure external knowledge as a graph with entities and relationships, then query that graph to provide the LLM with highly structured, relevant context before gen

In [23]:
count_tokens(
    conversation_bufw,
    "I just want to analyze the different possibilities. What can you think of?"
)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: My interest here is to explore the potential of integrating Large Language Models with external knowledge
AI: Oh, that's a fascinating area! I'm genuinely excited to discuss this with you—it's one of the most active and important frontiers in AI right now.

There are several really compelling approaches to integrating LLMs with external knowledge:

**Retrieval-Augmented Generation (RAG)** is probably the most popular approach. The basic idea is that when you query an LLM, it first retrieves relevant documents or data from an external knowledge base, then uses those retrieved chunks to inform its response. This helps ground the model's outputs in factual, up-to-d

"Great! Let me map out a pretty comprehensive set of possibilities for you:\n\n**Data Integration Approaches:**\n- **Structured Data Sources**: SQL databases, data warehouses, APIs that return JSON/XML—the LLM can learn to query these directly or have results fed to it\n- **Unstructured Data**: Document collections, PDFs, web pages, research papers that get indexed and retrieved\n- **Semi-structured Data**: CSVs, logs, configuration files that sit somewhere between structured and unstructured\n- **Real-time Streams**: Live data feeds, stock prices, weather data, social media—keeping knowledge current rather than static\n- **Private vs. Public Knowledge**: Internal company documents versus publicly available internet data\n\n**Architectural Patterns:**\n- **Synchronous Retrieval**: Look up knowledge right before generating (RAG)\n- **Asynchronous Indexing**: Pre-process and embed knowledge ahead of time so retrieval is fast\n- **Hybrid Approaches**: Some knowledge embedded in the model,

In [ ]:
count_tokens(
    conversation_bufw,
    "Which data source types could be used to give context to the model?"
)

Spent a total of 337 tokens


"There are various data source types that can be used to give context to a model. Some common examples include text corpora, knowledge graphs, databases, ontologies, and even social media feeds. These data sources can provide a wealth of information that can help the model better understand and interpret the input it receives. Additionally, domain-specific data sources such as scientific literature, financial reports, or medical records can also be used to provide context and enhance the model's performance in specific areas. Ultimately, the choice of data sources will depend on the specific task and domain in which the model is being applied."

In [ ]:
count_tokens(
    conversation_bufw,
    "What is my aim again?"
)

Spent a total of 262 tokens


"Your aim is to understand the different data source types that can be used to give context to a model in order to enhance its performance and accuracy. By utilizing various data sources effectively, you can improve the model's ability to interpret and analyze input data in a specific domain or task."

By the end of the conversation, when we ask **"What is my aim again?"**, the answer to this was contained in the human response three interactions ago. As we only kept the most recent interaction (`k=1`), the model had forgotten and could not give the correct answer.

We can see the effective “memory” of the model like so:

In [ ]:
bufw_history = conversation_bufw.memory.load_memory_variables(
    inputs=[]
)['history']

In [ ]:
print(bufw_history)

Human: What is my aim again?
AI: Your aim is to understand the different data source types that can be used to give context to a model in order to enhance its performance and accuracy. By utilizing various data sources effectively, you can improve the model's ability to interpret and analyze input data in a specific domain or task.


Although this method isn’t suitable for remembering distant interactions, it is good at limiting the number of tokens being used — a number that we can increase/decrease depending on our needs. For the longer conversation used in our earlier comparison, we can set `k=6` and reach ~1.5K tokens per interaction after 27 total interactions:

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/conversation_bw.webp" alt='auto' width="1000"/>
</div>

Token count including the ConversationBufferWindowMemory at k=6 and k=12.

If we only need memory of recent interactions, this is a great option. However, for a mix of both distant and recent interactions, there are other options.

### ConversationSummaryBufferMemory
The `ConversationSummaryBufferMemory` is a mix of the `ConversationSummaryMemory` and the `ConversationBufferWindowMemory`. It summarizes the earliest interactions in a conversation while maintaining the max_token_limit most recent tokens in their conversation. It is initialized like so:

In [25]:
conversation_sum_bufw = ConversationChain(
    llm=llm, memory=ConversationSummaryBufferMemory(
        llm=llm,
        verbose=True,
        max_token_limit=400
))

When applying this to our earlier conversation, we can set `max_token_limit` to a small number and yet the LLM can remember our earlier “aim”.

This is because that information is captured by the “summarization” component of the memory, despite being missed by the “buffer window” component.

Naturally, the pros and cons of this component are a mix of the earlier components on which this is based.

<!DOCTYPE html>
<html>
<head>
    <style>
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th, td {
            border: 1px solid black;
            padding: 10px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
</head>
<body>

<table>
    <tr>
        <th>Pros</th>
        <th>Cons</th>
    </tr>
    <tr>
        <td>Summarizer means we can remember distant interactions</td>
        <td>Summarizer increases token count for shorter conversations</td>
    </tr>
    <tr>
        <td>Buffer prevents us from missing information from the most recent interactions</td>
        <td>Storing the raw interactions — even if just the most recent interactions — increases token count</td>
    </tr>
</table>

</body>
</html>


Although requiring more tweaking on what to summarize and what to maintain within the buffer window, the `ConversationSummaryBufferMemory` does give us plenty of flexibility and is the only one of our memory types (so far) that allows us to remember distant interactions and store the most recent interactions in their raw — and most information-rich — form.

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-memory-rag/memory_bws.webp" alt='auto' width="1000"/>
</div>

Token count comparisons including the ConversationSummaryBufferMemory type with max_token_limit values of 650 and 1300.

We can also see that despite including a summary of past interactions and the raw form of recent interactions — the increase in token count of `ConversationSummaryBufferMemory` is competitive with other methods.

### Other Memory Types
The memory types we have covered here are great for getting started and give a good balance between remembering as much as possible and minimizing tokens.

However, we have other options — particularly the `ConversationKnowledgeGraphMemory` and `ConversationEntityMemory`. We’ll give these different forms of memory the attention they deserve in upcoming chapters.

That’s it for this introduction to conversational memory for LLMs using LangChain. As we’ve seen, there are plenty of options for helping stateless LLMs interact as if they were in a stateful environment — able to consider and refer back to past interactions.

As mentioned, there are other forms of memory we can cover. We can also implement our own memory modules, use multiple types of memory within the same chain, combine them with agents, and much more. All of which we will cover in the future.